# CIFAR-100 Long-Tail (LT) 불균형 분류 실험

---

## 1. 태스크 및 도메인
- **도메인**: CIFAR-100 Long-Tail (인공 불균형) 이미지 분류
- **모달리티**: RGB 컬러 이미지 (32×32)
- **태스크**: 100-class 분류
  - 클래스: 동물(50개), 객체(50개) 계층 구조
- **핵심 도전**: CIFAR-10보다 훨씬 극단적 불균형 (IR=100 시 min=5 샘플/클래스)

## 2. 모델
- **아키텍처**: ResNet-32 (He et al. 2016 CIFAR 전용)
- **사전학습**: 없음 (CIFAR-LT 표준)
- **선택 이유**: CIFAR-LT 벤치마크 표준 모델
- **출력**: 100채널 softmax logits

## 3. 데이터셋
- **이름**: CIFAR-100 Long-Tail (torchvision + 지수 감소 서브샘플링)
- **규모**: 
  - Original CIFAR-100: 50,000 train / 10,000 test (클래스당 500 / 100)
  - LT 변환: IR=100 → train 클래스당 500~5장, test 균형 유지
- **입력 해상도**: 32×32 RGB
- **클래스 불균형**: 
  - IR=10: max 500 / min 50 (비교적 완만)
  - IR=50: max 500 / min 10
  - IR=100: max 500 / min 5 (매우 극심 — few-shot 검증에 최적)
- **공식 분할**: 없음 → 8:1:1 (train 40,000 / val 5,000 / test 10,000)

## 4. 데이터 준비 (협업자용)
> Cell 0 자동 실행 시 torchvision으로 자동 다운로드됩니다.

**취득 방법**:
- `torchvision.datasets.CIFAR100(download=True)` — Cell 0 실행 시 자동 다운로드

**Colab 환경**: 설치 불필요

## 5. 전처리 및 데이터 특이점
- **Augmentation (학습)**: RandomCrop(32, padding=4) + RandomHorizontalFlip
- **정규화**: ImageNet 기준 mean/std 사용
  - mean=[0.5071, 0.4867, 0.4408]
  - std=[0.2675, 0.2565, 0.2761]
- **불균형 생성**: 지수 감소 분포 — n_i = n_max × IR^(-i/(K-1))
  - n_max = 500 (원본)
  - K = 100 (클래스 수)
- **Test Set**: 원본 균형 유지

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce` | — | — | — |
| `wce` | — | — | — |
| `lwce` | — | — | — |
| `plwce` | alpha | 2.5 ~ 15.0 | 20 (1D GridSampler) |
| `cb` | — | — | — |
| `plwce_focal` | alpha + gamma | alpha 2.5~15.0(8) × gamma 0.5~5.0(5) | 40 (2D GridSampler) |

**Optuna 설정** (proxy learning):
- subset_ratio=0.20
- proxy_epochs=40
- metric: Balanced Accuracy

## 7. SoTA 참고 (2025년 12월 기준, CIFAR-100 LT)
| 방법 | Top-1 Acc (%) | Balanced Acc (%) | Few-shot Acc (%) | 출처 |
|------|-------------|-----------------|------------------|------|
| Decoupling (2019) | 47.69 (IR=100) | — | — | ICCV'19 |
| cRT (2020) | 50.68 (IR=100) | — | — | ICML'21 |
| BBN (2019) | 45.51 (IR=100) | — | — | ICCV'19 |
| LDAM (2019) | 41.99 (IR=100) | — | — | ICML'19 |

> 본 연구 목표: ResNet-32 + 표준 SGD 학습 하에서 LWCE/PLWCE 손실함수의 효과 검증 (극단적 불균형 환경).
> 평가 지표: Top-1 정확도 + Balanced Accuracy + Few-shot Accuracy


In [5]:
# === Cell 0: 환경 설정 ===

!pip install optuna torchvision pandas openpyxl -q

import os, sys, json, pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from sklearn.metrics import confusion_matrix, f1_score

import optuna
from optuna.samplers import GridSampler

# --- Google Drive 마운트 (Colab only) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('✓ Google Drive 마운트 완료')
except:
    IN_COLAB = False
    print('✓ 로컬/하이브리드 환경에서 실행 중')

# --- 모듈 경로 설정: custom_losses.py와 resnet32.py 찾기 ---
# Case 1: 현재 폴더에 있는 경우 (로컬 또는 Colab 업로드)
IMG_CLF_DIR = os.getcwd()
if not os.path.exists(f'{IMG_CLF_DIR}/custom_losses.py'):
    # Case 2: Google Drive에서 찾기 (Colab)
    if IN_COLAB:
        IMG_CLF_DIR = '/content/drive/MyDrive/imbalanced-data-LWCE/image_classification'
    # Case 3: 로컬 경로
    else:
        IMG_CLF_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() \
                      else 'C:/Users/Seung/Desktop/Research/Deep_Learning/imbalanced-data-LWCE/image_classification'

if IMG_CLF_DIR not in sys.path:
    sys.path.insert(0, IMG_CLF_DIR)

# 모듈 임포트 확인
try:
    from custom_losses import get_clf_loss
    from resnet32 import build_resnet32
    print(f'✓ 모듈 로드 성공: {IMG_CLF_DIR}')
except ModuleNotFoundError as e:
    print(f'⚠️  모듈 로드 실패: {e}')
    print(f'   찾는 경로: {IMG_CLF_DIR}')
    print(f'   Colab: {IN_COLAB}')
    raise

# --- 상수 설정 ---
DATASET = 'cifar100'
NUM_CLASSES = 100
IR_LIST = [10, 50, 100]

BATCH_SIZE = 128
NUM_WORKERS = 0
SEED = 42
FINAL_EPOCHS = 200

# 결과 저장 경로
if IN_COLAB:
    RESULTS_BASE = '/content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR100_LT'
else:
    RESULTS_BASE = './results/CIFAR100_LT'

os.makedirs(RESULTS_BASE, exist_ok=True)

# --- 디바이스 설정 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# --- 랜덤 시드 고정 ---
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

# --- Matplotlib 백엔드 ---
matplotlib.use('Agg')

print('\n✓ 환경 설정 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive 마운트 완료
✓ 모듈 로드 성공: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification
✓ Device: cuda
  GPU: NVIDIA A100-SXM4-80GB
  Memory: 85.1 GB

✓ 환경 설정 완료


In [6]:
# === Cell 1: CIFAR-100 LT 데이터셋 생성 및 로드 ===

def make_cifar_lt(dataset_name: str, imbalance_ratio: int, seed: int = 42):
    """
    CIFAR 데이터셋을 불균형(long-tail) 분포로 변환.
    지수 감소: n_i = n_max × IR^(-i/(K-1))
    """
    K = 10 if dataset_name == 'cifar10' else 100
    n_max = 5000 if dataset_name == 'cifar10' else 500
    
    # 전체 데이터셋 다운로드
    if dataset_name == 'cifar10':
        dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    else:
        dataset = datasets.CIFAR100(root='/tmp/cifar', train=True, download=True, transform=None)
    
    targets = np.array(dataset.targets)
    
    # 클래스별 인덱스 그룹화
    class_indices = [np.where(targets == c)[0] for c in range(K)]
    
    # 불균형 수정: n_i 계산
    rho = imbalance_ratio ** (-1 / (K - 1))
    class_counts = [int(n_max * (rho ** i)) for i in range(K)]
    
    # 각 클래스에서 n_i개씩 랜덤 선택
    np.random.seed(seed)
    lt_indices = []
    for c, n_samples in enumerate(class_counts):
        n_samples = max(1, n_samples)
        selected = np.random.choice(class_indices[c], size=n_samples, replace=False)
        lt_indices.extend(selected)
    
    lt_indices = np.array(lt_indices)
    np.random.shuffle(lt_indices)
    
    return lt_indices.tolist(), class_counts


def load_cifar_lt_loaders(ir: int, batch_size: int = 128, num_workers: int = 0):
    """
    CIFAR-100 LT + standard test을 로드.
    Train LT set을 80/20으로 나눔.
    """
    # LT train 생성
    full_dataset = datasets.CIFAR100(root='/tmp/cifar', train=True, download=True, transform=None)
    lt_indices, class_counts = make_cifar_lt('cifar100', ir, seed=SEED)
    lt_indices = np.array(lt_indices)  # Convert to numpy array for proper indexing
    
    # train/val 분할 (stratified)
    lt_targets = np.array(full_dataset.targets)[lt_indices]
    
    # 클래스별로 stratified split
    train_indices, val_indices = [], []
    for c in range(100):
        c_mask = lt_targets == c
        c_idx = np.where(c_mask)[0]
        if len(c_idx) == 0:
            continue
        np.random.seed(SEED)
        np.random.shuffle(c_idx)
        n_c_val = max(1, len(c_idx) // 5)
        val_indices.extend(lt_indices[c_idx[:n_c_val]])
        train_indices.extend(lt_indices[c_idx[n_c_val:]])
    
    # Transform 설정 (CIFAR-100 정규화)
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071, 0.4867, 0.4408],
                             std=[0.2675, 0.2565, 0.2761]),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5071, 0.4867, 0.4408],
                             std=[0.2675, 0.2565, 0.2761]),
    ])
    
    # Datasets with transform
    train_ds = Subset(full_dataset, train_indices)
    train_ds.dataset.transform = train_tf
    
    val_ds = Subset(full_dataset, val_indices)
    val_ds.dataset.transform = test_tf
    
    test_ds = datasets.CIFAR100(root='/tmp/cifar', train=False, download=True, transform=test_tf)
    
    # DataLoaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    return train_loader, val_loader, test_loader, class_counts


print('✓ CIFAR-100 LT 함수 정의 완료')

✓ CIFAR-100 LT 함수 정의 완료


In [7]:
# === Cell 2: 클래스 분포 시각화 ===

def visualize_class_distribution(class_counts, ir, dataset_name='CIFAR-100'):
    counts_arr = np.array(class_counts)
    
    print(f'\n{'='*60}')
    print(f'{dataset_name} LT (IR={ir}) — 클래스 분포')
    print(f'{'='*60}')
    print(f'Min: {counts_arr.min():5d} | Max: {counts_arr.max():5d} | Ratio: {counts_arr.max()/counts_arr.min():.1f}:1')
    print(f'Total samples: {counts_arr.sum():,}')
    
    # Many/Medium/Few 그룹
    many_mask = counts_arr >= 100
    medium_mask = (counts_arr >= 20) & (counts_arr < 100)
    few_mask = counts_arr < 20
    
    print(f'\nGroup distribution:')
    print(f'  Many-shot (n≥100):   {many_mask.sum():3d} classes')
    print(f'  Medium-shot (20≤n):  {medium_mask.sum():3d} classes')
    print(f'  Few-shot (n<20):     {few_mask.sum():3d} classes')
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(14, 4))
    colors = ['green' if m else ('orange' if med else 'red') 
              for m, med in zip(many_mask, medium_mask)]
    ax.bar(range(len(class_counts)), class_counts, color=colors, alpha=0.7)
    ax.set_xlabel('Class')
    ax.set_ylabel('# Samples (log scale)', fontsize=11)
    ax.set_yscale('log')
    ax.set_title(f'{dataset_name} LT Distribution (IR={ir})', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_BASE}/IR{ir}/class_distribution.png', dpi=100, bbox_inches='tight')
    plt.close()
    print(f'\n✓ 분포 저장: {RESULTS_BASE}/IR{ir}/class_distribution.png')


for ir in IR_LIST:
    os.makedirs(f'{RESULTS_BASE}/IR{ir}', exist_ok=True)
    _, _, _, class_counts = load_cifar_lt_loaders(ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    visualize_class_distribution(class_counts, ir, 'CIFAR-100')


CIFAR-100 LT (IR=10) — 클래스 분포
Min:    49 | Max:   500 | Ratio: 10.2:1
Total samples: 19,572

Group distribution:
  Many-shot (n≥100):    70 classes
  Medium-shot (20≤n):   30 classes
  Few-shot (n<20):       0 classes

✓ 분포 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR100_LT/IR10/class_distribution.png

CIFAR-100 LT (IR=50) — 클래스 분포
Min:     9 | Max:   500 | Ratio: 55.6:1
Total samples: 12,607

Group distribution:
  Many-shot (n≥100):    41 classes
  Medium-shot (20≤n):   41 classes
  Few-shot (n<20):      18 classes

✓ 분포 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR100_LT/IR50/class_distribution.png

CIFAR-100 LT (IR=100) — 클래스 분포
Min:     5 | Max:   500 | Ratio: 100.0:1
Total samples: 10,847

Group distribution:
  Many-shot (n≥100):    35 classes
  Medium-shot (20≤n):   35 classes
  Few-shot (n<20):      30 classes

✓ 분포 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR100_LT/

In [8]:
# === Cell 3: 모델 및 평가 함수 정의 ===

def compute_val_metrics(model, loader, num_classes, class_counts_train=None,
                        group_thresholds=(100, 20)):
    model.eval()
    cm = torch.zeros(num_classes, num_classes, dtype=torch.long)
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            for t, p in zip(labels.view(-1), preds.view(-1)):
                cm[t.long(), p.long()] += 1
    
    # Per-class accuracy
    per_class_acc = (cm.diagonal().float() / cm.sum(1).clamp(min=1).float()).cpu().numpy()
    balanced_acc = float(per_class_acc.mean())
    top1_acc = float(cm.diagonal().sum() / cm.sum())
    
    # F1-Macro
    y_true, y_pred = [], []
    for i in range(num_classes):
        for j in range(num_classes):
            y_true.extend([i] * cm[i, j].item())
            y_pred.extend([j] * cm[i, j].item())
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    result = {
        'Top1_Acc': top1_acc,
        'Balanced_Acc': balanced_acc,
        'F1_Macro': f1_macro,
        'Per_Class_Acc': per_class_acc.tolist(),
    }
    
    if class_counts_train is not None:
        many_th, med_th = group_thresholds
        counts = np.array(class_counts_train)
        many_idx = np.where(counts >= many_th)[0]
        medium_idx = np.where((counts >= med_th) & (counts < many_th))[0]
        few_idx = np.where(counts < med_th)[0]
        
        result['Many_Acc'] = float(per_class_acc[many_idx].mean()) if len(many_idx) > 0 else 0.0
        result['Medium_Acc'] = float(per_class_acc[medium_idx].mean()) if len(medium_idx) > 0 else 0.0
        result['Few_Acc'] = float(per_class_acc[few_idx].mean()) if len(few_idx) > 0 else 0.0
    
    return result


def compute_val_acc(model, loader):
    model.eval()
    correct, total, class_correct, class_total = 0, 0, None, None
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(dim=1)
            
            if class_correct is None:
                class_correct = torch.zeros(NUM_CLASSES, dtype=torch.long)
                class_total = torch.zeros(NUM_CLASSES, dtype=torch.long)
            
            for c in range(NUM_CLASSES):
                mask = labels == c
                class_correct[c] += (preds[mask] == c).sum().item()
                class_total[c] += mask.sum().item()
    
    per_class_acc = (class_correct.float() / class_total.clamp(min=1).float()).numpy()
    return float(per_class_acc.mean())


print('✓ 모델 및 평가 함수 정의 완료')

✓ 모델 및 평가 함수 정의 완료


In [9]:
# === Cell 4: train_model 함수 정의 ===

def train_model(loss_name: str,
                class_counts: list,
                train_loader,
                val_loader,
                num_classes: int,
                alpha: float = 1.0,
                gamma: float = 2.0,
                epochs: int = 200,
                lr: float = 0.1,
                tag: str = ''):
    
    model = build_resnet32(num_classes).to(device)
    
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=2e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[160, 180], gamma=0.01)
    
    criterion = get_clf_loss(loss_name, class_counts, alpha=alpha, gamma=gamma)
    
    best_val_acc = 0.0
    best_model_state = None
    history = {'epoch': [], 'train_loss': [], 'val_acc': [], 'val_balanced_acc': []}
    
    pbar = tqdm(range(epochs), desc=f'{loss_name} (α={alpha:.2f}, γ={gamma:.2f})', leave=False)
    
    for epoch in pbar:
        # Train
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * labels.size(0)
        
        train_loss /= len(train_loader.dataset)
        
        # Validation
        val_bal_acc = compute_val_acc(model, val_loader)
        
        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_balanced_acc'].append(val_bal_acc)
        
        # Checkpoint
        if val_bal_acc > best_val_acc:
            best_val_acc = val_bal_acc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        scheduler.step()
        pbar.update()
    
    # Load best model
    if best_model_state:
        model.load_state_dict(best_model_state)
    model.eval()
    
    return model, history, best_val_acc


print('✓ train_model 함수 정의 완료')

✓ train_model 함수 정의 완료


In [10]:
# === Cell 5: Optuna alpha/gamma 탐색 ===

os.environ['TQDM_DISABLE'] = '1'

PROXY_EPOCHS = 40
PROXY_SUBSET_RATIO = 0.20
N_TRIALS_PWCE = 20      # pwce alpha 탐색
N_TRIALS_PLWCE = 20     # plwce alpha 탐색
N_TRIALS_FOCAL = 20     # focal gamma 탐색
ALPHA_LOW, ALPHA_HIGH = 2.0, 15.0
PWCE_LOW, PWCE_HIGH = 0.5, 5.0
GAMMA_LOW, GAMMA_HIGH = 0.5, 5.0

optuna_best = {}

print(f'Optuna 탐색 시작 (CIFAR-100, proxy: {PROXY_EPOCHS} epochs)')
print(f'pwce: alpha 범위 탐색 [0.5~5.0]')
print(f'plwce: alpha 범위 탐색 [2.0~15.0]')
print(f'focal: gamma 범위 탐색 [0.5~5.0]')
print(f'{"="*60}')

for ir in IR_LIST:
    print(f'\n[IR={ir}] Optuna 탐색 중...')
    
    train_loader, val_loader, _, class_counts = load_cifar_lt_loaders(
        ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    
    # Proxy subset
    n_subset = max(1, int(len(train_loader.dataset) * PROXY_SUBSET_RATIO))
    subset_indices = np.random.choice(len(train_loader.dataset), size=n_subset, replace=False)
    proxy_train_ds = Subset(train_loader.dataset, subset_indices)
    proxy_train_loader = DataLoader(proxy_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    
    # --- pwce alpha search ---
    def objective_pwce(trial):
        alpha = trial.suggest_float('alpha', PWCE_LOW, PWCE_HIGH)
        model, _, _ = train_model('pwce', class_counts, proxy_train_loader, val_loader,
                                   NUM_CLASSES, alpha=alpha, epochs=PROXY_EPOCHS, tag=f'optuna_pwce')
        return compute_val_acc(model, val_loader)
    
    # pwce: 초기값 [0.5, 1.0] + 범위 탐색 alpha
    pwce_alphas = [0.5, 1.0] + np.linspace(PWCE_LOW, PWCE_HIGH, N_TRIALS_PWCE-2).tolist()
    sampler_pwce = optuna.samplers.GridSampler({'alpha': pwce_alphas})
    study_pwce = optuna.create_study(direction='maximize', sampler=sampler_pwce,
                                      study_name=f'cifar100_ir{ir}_pwce')
    study_pwce.optimize(objective_pwce, n_trials=N_TRIALS_PWCE, show_progress_bar=False)
    
    # --- plwce alpha search ---
    def objective_plwce(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        model, _, _ = train_model('plwce', class_counts, proxy_train_loader, val_loader,
                                   NUM_CLASSES, alpha=alpha, epochs=PROXY_EPOCHS, tag=f'optuna_plwce')
        return compute_val_acc(model, val_loader)
    
    # plwce: 초기값 [0.5, 1.0] + 범위 탐색 alpha
    plwce_alphas = [0.5, 1.0] + np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS_PLWCE-2).tolist()
    sampler_plwce = optuna.samplers.GridSampler({'alpha': plwce_alphas})
    study_plwce = optuna.create_study(direction='maximize', sampler=sampler_plwce,
                                      study_name=f'cifar100_ir{ir}_plwce')
    study_plwce.optimize(objective_plwce, n_trials=N_TRIALS_PLWCE, show_progress_bar=False)
    
    # --- focal gamma search ---
    def objective_focal(trial):
        gamma = trial.suggest_float('gamma', GAMMA_LOW, GAMMA_HIGH)
        model, _, _ = train_model('focal', class_counts, proxy_train_loader, val_loader,
                                   NUM_CLASSES, gamma=gamma, epochs=PROXY_EPOCHS, tag=f'optuna_focal')
        return compute_val_acc(model, val_loader)
    
    # focal: gamma 범위 탐색
    focal_gammas = np.linspace(GAMMA_LOW, GAMMA_HIGH, N_TRIALS_FOCAL).tolist()
    sampler_focal = optuna.samplers.GridSampler({'gamma': focal_gammas})
    study_focal = optuna.create_study(direction='maximize', sampler=sampler_focal,
                                      study_name=f'cifar100_ir{ir}_focal')
    study_focal.optimize(objective_focal, n_trials=N_TRIALS_FOCAL, show_progress_bar=False)
    
    optuna_best[ir] = {
        'pwce': {'alpha': study_pwce.best_params['alpha']},
        'plwce': {'alpha': study_plwce.best_params['alpha']},
        'focal': {'gamma': study_focal.best_params['gamma']},
    }
    
    print(f'  PWCE best α={optuna_best[ir]["pwce"]["alpha"]:.3f}')
    print(f'  PLWCE best α={optuna_best[ir]["plwce"]["alpha"]:.3f}')
    print(f'  Focal best γ={optuna_best[ir]["focal"]["gamma"]:.3f}')
    
    os.makedirs(f'{RESULTS_BASE}/IR{ir}', exist_ok=True)
    with open(f'{RESULTS_BASE}/IR{ir}/optuna_results.json', 'w') as f:
        json.dump(optuna_best[ir], f, indent=2)

os.environ['TQDM_DISABLE'] = '0'
print(f'\n✓ Optuna 탐색 완료 — pwce, plwce, focal 최적화됨')
print(f'  (ce, lwce, cb는 기본값 사용)')

Optuna 탐색 시작 (CIFAR-100, proxy: 40 epochs)
pwce: alpha 범위 탐색 [0.5~5.0]
plwce: alpha 범위 탐색 [2.0~15.0]
focal: gamma 범위 탐색 [0.5~5.0]

[IR=10] Optuna 탐색 중...


[I 2026-04-27 08:43:52,948] A new study created in memory with name: cifar100_ir10_pwce


pwce (α=4.74, γ=2.00):   0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-04-27 08:46:22,157] Trial 0 finished with value: 0.03326923027634621 and parameters: {'alpha': 4.735294117647059}. Best is trial 0 with value: 0.03326923027634621.


pwce (α=1.00, γ=2.00):   0%|          | 0/40 [00:00<?, ?it/s]

...

focal (α=1.00, γ=4.05):   0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-04-27 14:06:53,782] Trial 18 finished with value: 0.08753055334091187 and parameters: {'gamma': 4.052631578947368}. Best is trial 10 with value: 0.11045397073030472.


focal (α=1.00, γ=3.34):   0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-04-27 14:08:08,616] Trial 19 finished with value: 0.09811875224113464 and parameters: {'gamma': 3.3421052631578947}. Best is trial 10 with value: 0.11045397073030472.


  PWCE best α=0.500
  PLWCE best α=2.000
  Focal best γ=1.684

✓ Optuna 탐색 완료 — pwce, plwce, focal 최적화됨
  (ce, lwce, cb는 기본값 사용)


In [11]:
# === Cell 6: 전체 Loss × IR 비교 실험 ===

LOSS_CONFIGS = ['ce', 'pwce', 'lwce', 'plwce', 'cb', 'focal']

all_results = {}
all_histories = {}

print(f'Full experiment: {len(IR_LIST)} IRs × {len(LOSS_CONFIGS)} losses = {len(IR_LIST)*len(LOSS_CONFIGS)} runs')
print(f'{'='*60}')

for ir in IR_LIST:
    print(f'\n[IR={ir}] 훈련 중...')
    
    train_loader, val_loader, test_loader, class_counts = load_cifar_lt_loaders(
        ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    
    all_results[ir] = {}
    all_histories[ir] = {}
    
    for loss_name in LOSS_CONFIGS:
        # Get alpha/gamma from Optuna
        alpha = optuna_best[ir].get(loss_name, {}).get('alpha', 1.0)
        gamma = optuna_best[ir].get('focal', {}).get('gamma', 2.0) if loss_name == 'focal' else 2.0
        
        # Train
        model, history, _ = train_model(loss_name, class_counts, train_loader, val_loader,
                                         NUM_CLASSES, alpha=alpha, gamma=gamma, epochs=FINAL_EPOCHS,
                                         tag=f'ir{ir}_final')
        
        # Evaluate on test set
        metrics = compute_val_metrics(model, test_loader, NUM_CLASSES, class_counts_train=class_counts)
        metrics['alpha'] = alpha
        metrics['gamma'] = gamma
        
        all_results[ir][loss_name] = metrics
        all_histories[ir][loss_name] = history
        
        print(f'  {loss_name:15s} | Top1={metrics["Top1_Acc"]:.4f} | Balanced={metrics["Balanced_Acc"]:.4f} | '
              f'Few={metrics.get("Few_Acc", 0):.4f}')

print(f'\n✓ 전체 훈련 완료')

Full experiment: 3 IRs × 6 losses = 18 runs

[IR=10] 훈련 중...


ce (α=1.00, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  ce              | Top1=0.3734 | Balanced=0.3734 | Few=0.0000


pwce (α=0.50, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  pwce            | Top1=0.3763 | Balanced=0.3763 | Few=0.0000


...

  plwce           | Top1=0.2150 | Balanced=0.2150 | Few=0.0380


cb (α=1.00, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  cb              | Top1=0.1975 | Balanced=0.1975 | Few=0.0413


focal (α=1.00, γ=1.68):   0%|          | 0/200 [00:00<?, ?it/s]

  focal           | Top1=0.2223 | Balanced=0.2223 | Few=0.0360

✓ 전체 훈련 완료


In [12]:
# === Cell 7: 결과 시각화 및 저장 ===

print(f'\n최종 결과 요약')
print(f'{'='*90}')

for ir in IR_LIST:
    print(f'\nIR={ir}:')
    print(f'{"Loss":15s} | {"Top1":>7s} | {"Balanced":>8s} | {"F1-Macro":>8s} | '
          f'{"Many":>7s} | {"Medium":>7s} | {"Few":>7s}')
    print('-' * 90)
    
    for loss_name in LOSS_CONFIGS:
        m = all_results[ir][loss_name]
        print(f'{loss_name:15s} | {m["Top1_Acc"]:7.4f} | {m["Balanced_Acc"]:8.4f} | '
              f'{m["F1_Macro"]:8.4f} | {m.get("Many_Acc", 0):7.4f} | '
              f'{m.get("Medium_Acc", 0):7.4f} | {m.get("Few_Acc", 0):7.4f}')
    
    # JSON 저장
    with open(f'{RESULTS_BASE}/IR{ir}/results.json', 'w') as f:
        json.dump(all_results[ir], f, indent=2)
    
    # Excel 저장
    summary_data = []
    for loss_name in LOSS_CONFIGS:
        m = all_results[ir][loss_name]
        summary_data.append({
            'Loss': loss_name,
            'Top1_Acc': f"{m['Top1_Acc']:.4f}",
            'Balanced_Acc': f"{m['Balanced_Acc']:.4f}",
            'F1_Macro': f"{m['F1_Macro']:.4f}",
            'Many_Acc': f"{m.get('Many_Acc', 0):.4f}",
            'Medium_Acc': f"{m.get('Medium_Acc', 0):.4f}",
            'Few_Acc': f"{m.get('Few_Acc', 0):.4f}",
            'Alpha': f"{m['alpha']:.3f}",
            'Gamma': f"{m['gamma']:.3f}",
        })
    
    summary_df = pd.DataFrame(summary_data)
    
    with pd.ExcelWriter(f'{RESULTS_BASE}/IR{ir}/results.xlsx', engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        
        # Training history
        history_data = []
        for loss_name in LOSS_CONFIGS:
            h = all_histories[ir][loss_name]
            for epoch, loss, val_acc in zip(h['epoch'], h['train_loss'], h['val_balanced_acc']):
                history_data.append({
                    'Loss': loss_name,
                    'Epoch': epoch,
                    'Train_Loss': f"{loss:.4f}",
                    'Val_Balanced_Acc': f"{val_acc:.4f}",
                })
        history_df = pd.DataFrame(history_data)
        history_df.to_excel(writer, sheet_name='Training_History', index=False)

# --- 학습 곡선 시각화 ---
fig, axes = plt.subplots(1, len(IR_LIST), figsize=(15, 4))
if len(IR_LIST) == 1:
    axes = [axes]

for ax_idx, ir in enumerate(IR_LIST):
    ax = axes[ax_idx]
    for loss_name in LOSS_CONFIGS:
        h = all_histories[ir][loss_name]
        ax.plot(h['epoch'], h['val_balanced_acc'], label=loss_name, alpha=0.7)
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation Balanced Accuracy')
    ax.set_title(f'IR={ir}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/training_curves_all_irs.png', dpi=100, bbox_inches='tight')
plt.close()
print(f'\n✓ 훈련 곡선 저장: {RESULTS_BASE}/training_curves_all_irs.png')

print(f'\n✓ 모든 결과 저장 완료')
print(f'  JSON: {RESULTS_BASE}/IR*/results.json')
print(f'  Excel: {RESULTS_BASE}/IR*/results.xlsx')
print(f'  Plots: {RESULTS_BASE}/*.png')


최종 결과 요약

IR=10:
Loss            |    Top1 | Balanced | F1-Macro |    Many |  Medium |     Few
------------------------------------------------------------------------------------------
ce              |  0.3734 |   0.3734 |   0.3606 |  0.4410 |  0.2157 |  0.0000
pwce            |  0.3763 |   0.3763 |   0.3633 |  0.4466 |  0.2123 |  0.0000
lwce            |  0.3818 |   0.3818 |   0.3719 |  0.4427 |  0.2397 |  0.0000
plwce           |  0.3915 |   0.3915 |   0.3824 |  0.4534 |  0.2470 |  0.0000
cb              |  0.3616 |   0.3616 |   0.3486 |  0.4316 |  0.1983 |  0.0000
focal           |  0.3693 |   0.3693 |   0.3562 |  0.4413 |  0.2013 |  0.0000

IR=50:
Loss            |    Top1 | Balanced | F1-Macro |    Many |  Medium |     Few
------------------------------------------------------------------------------------------
ce              |  0.2579 |   0.2579 |   0.2287 |  0.4159 |  0.1829 |  0.0689
pwce            |  0.2596 |   0.2596 |   0.2290 |  0.4188 |  0.1905 |  0.0544
lwce        